##### Import the libraries

In [44]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import holidays

##### Load the datasets

In [45]:
ad = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\Cleaned\admission_discharge_cleaned.csv")
icu = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\TimeSeries\ICU_TimeSeries.csv")
day_case = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\TimeSeries\Day_Case_Unit_TimeSeries.csv")
general_a = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\TimeSeries\General_Medicine_Ward_A_TimeSeries.csv")
general_b = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\TimeSeries\General_Medicine_Ward_B_TimeSeries.csv")
oncology = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\TimeSeries\Oncology_Ward_TimeSeries.csv")
ortho_a = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\TimeSeries\Orthopaedics_Ward_A_TimeSeries.csv")
ortho_b = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\TimeSeries\Orthopaedics_Ward_B_TimeSeries.csv")
cardiology = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\TimeSeries\Cardiology_Ward_TimeSeries.csv")

##### Create Dictionary of All Time-Series Datasets

In [46]:
units = {
    "ICU": icu,
    "Cardiology": cardiology,
    "Oncology": oncology,
    "Day_Case": day_case,
    "General_Medicine_A": general_a,
    "General_Medicine_B": general_b,
    "Orthopaedics_A": ortho_a,
    "Orthopaedics_B": ortho_b
}

##### Create daily LOS statistics

In [47]:
# Convert admission datetime
ad["admission_datetime"] = pd.to_datetime(ad["admission_datetime"])

# Create daily date
ad["datetime"] = ad["admission_datetime"].dt.floor("D")

los_daily = (ad.groupby(["hospital_id","ward","datetime"]).agg(avg_los=("length_of_stay_hours","mean"),
            median_los=("length_of_stay_hours","median"), max_los=("length_of_stay_hours","max")).reset_index())

los_daily.head()

,hospital_id,ward,datetime,avg_los,median_los,max_los
0,HHN-BIR-01,Cardiology Ward,2024-01-01,83.650000,69.20,143.8
1,HHN-BIR-01,Cardiology Ward,2024-01-02,101.810000,85.65,240.9
2,HHN-BIR-01,Cardiology Ward,2024-01-03,69.628571,55.60,133.6
3,HHN-BIR-01,Cardiology Ward,2024-01-04,66.457143,70.90,145.4
4,HHN-BIR-01,Cardiology Ward,2024-01-05,93.983333,93.85,144.7


##### UK public holiday

In [48]:
uk_holidays = holidays.UnitedKingdom(years=[2024,2025])

##### Feature Engineering

In [49]:
engineered_datasets = {}

for name, df in units.items():

    print(f"Processing {name}")
    df = df.copy()

    # Datetime
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.sort_values(["hospital_id","datetime"])

    # Lag Features
    df["lag_1"] = (df.groupby("hospital_id")["occupied_beds"].shift(1))
    df["lag_2"] = (df.groupby("hospital_id")["occupied_beds"].shift(2))
    df["lag_7"] = (df.groupby("hospital_id")["occupied_beds"].shift(7))

    # Rolling Mean
    df["rolling_mean_7"] = (df.groupby("hospital_id")["occupied_beds"].transform(lambda x: x.rolling(7).mean()))

    # Rolling Standard Deviation
    df["rolling_std_7"] = (df.groupby("hospital_id")["occupied_beds"].transform(lambda x: x.rolling(7).std()))

    # Calendar Features
    df["day_of_week"] = df["datetime"].dt.dayofweek
    df["month"] = df["datetime"].dt.month
    df["quarter"] = df["datetime"].dt.quarter
    df["week_of_year"] = ( df["datetime"].dt.isocalendar().week.astype(int))

    # Weekend
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    # Public Holiday
    df["is_public_holiday"] = (df["datetime"].dt.date.apply(lambda x: 1 if x in uk_holidays else 0))

    # Merge LOS
    df = df.merge(los_daily, on=["hospital_id","ward","datetime"], how="left")

    engineered_datasets[name] = df

print("Feature engineering completed.")

Processing ICU
Processing Cardiology
Processing Oncology
Processing Day_Case
Processing General_Medicine_A
Processing General_Medicine_B
Processing Orthopaedics_A
Processing Orthopaedics_B
Feature engineering completed.


    Lag features help the model learn that recent occupancy often influences future occupancy.
    Rolling statistics capture short-term trends and variability.
    Calendar features represent weekly and seasonal operational patterns that affect hospital activity.
    Public holidays can influence elective procedures, admissions, and staffing.

In [50]:
df.isnull().sum()

hospital_id            0
datetime               0
ward                   0
occupied_beds          0
staffed_beds           0
occupancy_rate         0
lag_1                  5
lag_2                 10
lag_7                 35
rolling_mean_7        30
rolling_std_7         30
day_of_week            0
month                  0
quarter                0
week_of_year           0
is_weekend             0
is_public_holiday      0
avg_los              178
median_los           178
max_los              178
dtype: int64

Fill mising LOA by Hospital + Ward

In [51]:
for col in ["avg_los", "median_los", "max_los"]:
    df[col] = (df.groupby(["hospital_id", "ward"])[col].transform(lambda x: x.fillna(x.mean())))

In [52]:
df = df.dropna(subset=["lag_1", "lag_2", "lag_7", "rolling_mean_7", "rolling_std_7"])

In [53]:
df.isnull().sum()

hospital_id          0
datetime             0
ward                 0
occupied_beds        0
staffed_beds         0
occupancy_rate       0
lag_1                0
lag_2                0
lag_7                0
rolling_mean_7       0
rolling_std_7        0
day_of_week          0
month                0
quarter              0
week_of_year         0
is_weekend           0
is_public_holiday    0
avg_los              0
median_los           0
max_los              0
dtype: int64

    Feature engineering introduced missing values in lag and rolling-window variables for the initial observations of each time series, as previous observations were unavailable. These rows (less than 1% of the dataset) were removed prior to modelling. Missing Length of Stay (LOS) statistics (4.87%) arose from unmatched admission records after merging occupancy and admissions data. These values were imputed by filling with mean by hospital and ward. This approach preserved clinically meaningful variation while retaining the maximum number of observations for model